In [ ]:
"""
Buy Bot — Telegram Poller → IBKR Paper Trading → MongoDB
Listens for NEW signals, validates EMA, places limit buy, records trade.

Fix: uses StringSession (no SQLite file lock) with the session string
persisted to SESSION_FILE so auth survives restarts without re-OTP.
Fix 2: forces UTF-8 on all log handlers so Windows cp1252 never sees emoji.
"""

import asyncio
import logging
import os
import sys
import threading
import re
from datetime import datetime, timezone
import random

import pandas as pd
from pymongo import MongoClient

from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import FloodWaitError, UserAlreadyParticipantError
from telethon.tl.functions.messages import ImportChatInviteRequest

from ib_insync import *

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ─── Logging  (UTF-8 safe: works in Jupyter AND Windows terminal) ─────────────
class _Utf8SafeStreamHandler(logging.StreamHandler):
    """
    Never raises UnicodeEncodeError.
    Works in Jupyter (no fileno), Windows cp1252, and normal shells.
    """
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass  # Jupyter OutStream has no reconfigure -- that is fine


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")

    sh = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)

    fh = logging.FileHandler("buy_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)

    logger = logging.getLogger("buy_bot")
    logger.setLevel(logging.INFO)
    # Clear so re-running the cell in Jupyter does not duplicate output
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ─── Telegram Credentials ─────────────────────────────────────────────────────
API_ID        = 34025130
API_HASH      = "e5e514ebcb8a15042feadfecd7a15cfe"
PHONE         = "+18477675748"
INVITE_LINK   = "https://t.me/+M5HRCxZycqJmOTg5"
POLL_INTERVAL = 20   # seconds

# StringSession avoids SQLite file-lock issues entirely.
# On first run it asks for your OTP and saves the session string here.
SESSION_FILE  = "buy_bot_session.txt"

# ─── IBKR Paper Trading ───────────────────────────────────────────────────────
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497        # TWS paper=7497 | IB Gateway paper=4002
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Trade Rules ──────────────────────────────────────────────────────────────
ORDER_SIZE = 10   # fixed shares

# ─── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client              = MongoClient("mongodb://localhost:27017/")
db                        = mongo_client["brkout_database"]
trades_collection         = db["trades"]
failed_entries_collection = db["failed_bad_entries"]
ticker_collection         = db["tickers"]
ticker_trade_collection   = db["tickers_trade"]


# ══════════════════════════════════════════════════════════════════════════════
# Telegram Parsers
# ══════════════════════════════════════════════════════════════════════════════
def extract_ticker(message_text: str) -> str | None:
    text = message_text.strip()
    m = re.match(r'^\[.*?\]\s+([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    m = re.match(r'^([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    return None


def parse_structured_message(text: str) -> dict | None:
    result = {"symbol": None, "status": None}
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^ticker\s*:", line):
            result["symbol"] = line.split(":", 1)[1].strip().upper()
        if re.match(r"(?i)^status\s*:", line):
            result["status"] = line.split(":", 1)[1].strip()
    if result["symbol"] and result["status"]:
        return result
    return None


def is_new_signal(status_text: str) -> bool:
    return "new" in status_text.lower()


# ══════════════════════════════════════════════════════════════════════════════
# Safe print — strips / replaces characters the console can't handle
# ══════════════════════════════════════════════════════════════════════════════
def safe_print(text: str):
    """
    Print to stdout, replacing any unencodable characters with '?'.
    Works in Jupyter (no .buffer) and normal terminals.
    """
    try:
        # Normal terminal: write bytes directly to avoid codec issues
        encoded = text.encode(sys.stdout.encoding or "utf-8", errors="replace")
        sys.stdout.buffer.write(encoded + b"\n")
        sys.stdout.buffer.flush()
    except AttributeError:
        # Jupyter: OutStream has no .buffer — just print, it handles encoding
        print(text.encode("utf-8", errors="replace").decode("utf-8", errors="replace"))
    except Exception:
        print(text)  # last resort


# ══════════════════════════════════════════════════════════════════════════════
# IBKR Client (Buy-side only)
# ══════════════════════════════════════════════════════════════════════════════
class IBKRBuyClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── Checks ────────────────────────────────────────────────────────────────
    def has_open_position(self, symbol: str) -> bool:
        return any(
            p.contract.symbol == symbol and p.position != 0
            for p in self.ib.positions()
        )

    def has_pending_order(self, symbol: str) -> bool:
        return any(t.contract.symbol == symbol for t in self.ib.openTrades())

    # ── Market Data ───────────────────────────────────────────────────────────
    def get_ask_price(self, symbol: str) -> float | None:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "", False, False)
        self.ib.sleep(2)
        self.ib.cancelMktData(contract)
        if ticker.ask and ticker.ask > 0:
            return float(ticker.ask)
        if ticker.last and ticker.last > 0:
            return float(ticker.last)
        return None

    def get_1min_bars(self, symbol: str, n: int = 70) -> pd.DataFrame | None:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        raw = self.ib.reqHistoricalData(
            contract,
            endDateTime="",
            durationStr="5 D",
            barSizeSetting="5 min",
            whatToShow="TRADES",
            useRTH=True,
        )
        if not raw:
            return None
        df = pd.DataFrame([
            {"high": b.high, "low": b.low, "close": b.close}
            for b in raw
        ]).tail(n).reset_index(drop=True)
        return df

    def ema_bullish(self, symbol: str) -> bool:
        try:
            df = self.get_1min_bars(symbol, n=70)
            if df is None or len(df) < 50:
                log.warning(f"{symbol}: not enough bars for EMA")
                return False
            ema21 = df["close"].ewm(span=21, adjust=False).mean().iloc[-1]
            ema50 = df["close"].ewm(span=50, adjust=False).mean().iloc[-1]
            log.info(f"{symbol}: EMA21={ema21:.4f}  EMA50={ema50:.4f}  bullish={ema21 > ema50}")
            return bool(ema21 > ema50)
        except Exception as e:
            log.error(f"{symbol}: EMA error - {e}")
            return False

    # ── Orders ────────────────────────────────────────────────────────────────
    def place_limit_buy(self, symbol: str, ask: float, qty: int = ORDER_SIZE):
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        trade = self.ib.placeOrder(contract, LimitOrder("BUY", qty, round(ask, 2)))
        self.ib.sleep(1)
        log.info(f"BUY {symbol} x{qty} @ ${ask:.2f}  orderId={trade.order.orderId}")
        return trade

    def get_avg_fill_price(self, symbol: str, order_id: int) -> float | None:
        self.ib.sleep(2)
        for trade in self.ib.trades():
            if (
                trade.contract.symbol == symbol
                and trade.order.orderId == order_id
                and trade.orderStatus.avgFillPrice > 0
            ):
                avg = float(trade.orderStatus.avgFillPrice)
                log.info(f"{symbol}: avg fill price from broker = ${avg:.4f}")
                return avg

        fills = self.ib.executions()
        symbol_fills = [
            f for f in fills
            if f.contract.symbol == symbol and f.execution.orderId == order_id
        ]
        if symbol_fills:
            total_qty  = sum(f.execution.shares for f in symbol_fills)
            total_cost = sum(f.execution.shares * f.execution.price for f in symbol_fills)
            avg = total_cost / total_qty
            log.info(f"{symbol}: avg fill price from executions = ${avg:.4f}")
            return avg

        log.warning(f"{symbol}: could not retrieve avg fill price for orderId={order_id}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# MongoDB Helpers
# ══════════════════════════════════════════════════════════════════════════════
def save_trade(symbol: str, limit_price: float, avg_fill_price: float | None,
               qty: int, order_id: int):
    entry_price = avg_fill_price if avg_fill_price else limit_price
    doc = {
        "symbol":          symbol,
        "limit_price":     limit_price,
        "entry_price":     entry_price,
        "avg_fill_price":  avg_fill_price,
        "qty":             qty,
        "order_id":        order_id,
        "status":          "open",
        "entered_at":      datetime.now(timezone.utc),
        "exit_price":      None,
        "exit_reason":     None,
        "exited_at":       None,
        "pnl_pct":         None,
    }
    trades_collection.insert_one(doc)
    ticker_trade_collection.update_one(
        {"symbol": symbol},
        {"$set": {"last_trade": doc, "updated_at": datetime.now(timezone.utc)}},
        upsert=True,
    )
    fill_str = f"${avg_fill_price:.4f}" if avg_fill_price else "pending"
    log.info(f"trade saved  {symbol}  limit=${limit_price:.2f}  avg_fill={fill_str}")


def save_failed_entry(symbol: str, reason: str):
    failed_entries_collection.insert_one({
        "symbol":    symbol,
        "reason":    reason,
        "timestamp": datetime.now(timezone.utc),
    })
    log.info(f"failed entry {symbol} -- {reason}")


# ══════════════════════════════════════════════════════════════════════════════
# Buy Logic
# ══════════════════════════════════════════════════════════════════════════════
def process_buy_signal(symbol: str, ibkr: IBKRBuyClient):
    log.info(f"--- BUY SIGNAL: {symbol} ---")

    if ibkr.has_open_position(symbol):
        save_failed_entry(symbol, "already_in_position"); return
    if ibkr.has_pending_order(symbol):
        save_failed_entry(symbol, "pending_order_exists"); return
    if not ibkr.ema_bullish(symbol):
        save_failed_entry(symbol, "ema_not_bullish"); return

    ask = ibkr.get_ask_price(symbol)
    if not ask:
        save_failed_entry(symbol, "no_ask_price"); return

    trade    = ibkr.place_limit_buy(symbol, ask, ORDER_SIZE)
    order_id = trade.order.orderId
    avg_fill = ibkr.get_avg_fill_price(symbol, order_id)

    save_trade(symbol, ask, avg_fill, ORDER_SIZE, order_id)
    ticker_collection.update_one(
        {"symbol": symbol},
        {"$set": {"last_signal": datetime.now(timezone.utc), "status": "bought"}},
        upsert=True,
    )


# ══════════════════════════════════════════════════════════════════════════════
# Session helpers
# ══════════════════════════════════════════════════════════════════════════════
def load_session_string() -> str:
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, encoding="utf-8") as f:
            return f.read().strip()
    return ""


def save_session_string(session_str: str):
    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(session_str)
    log.info(f"Session saved -> {SESSION_FILE}")


# ══════════════════════════════════════════════════════════════════════════════
# Telegram Poller
# ══════════════════════════════════════════════════════════════════════════════
async def poll_telegram(ibkr: IBKRBuyClient):
    while True:
        session_str = load_session_string()
        client = TelegramClient(StringSession(session_str), API_ID, API_HASH)
        try:
            await client.start(phone=PHONE)
            save_session_string(client.session.save())

            m = re.search(r't\.me/(?:joinchat/|\+)([A-Za-z0-9_-]+)', INVITE_LINK)
            if not m:
                raise ValueError(f"Cannot parse invite hash: {INVITE_LINK}")
            try:
                await client(ImportChatInviteRequest(m.group(1)))
                log.info("Joined channel")
            except UserAlreadyParticipantError:
                log.info("Already a member")

            channel = await client.get_entity(INVITE_LINK)
            log.info(f"Channel: {channel.title}  (ID: {channel.id})")
            safe_print("-" * 50)

            last_seen_id = None

            while True:
                messages = await client.get_messages(channel, limit=1)

                if messages:
                    msg = messages[0]
                    if msg.text:
                        is_new = (last_seen_id is None or msg.id != last_seen_id)

                        parsed = parse_structured_message(msg.text)
                        if parsed:
                            symbol = parsed["symbol"]
                            status = parsed["status"]
                        else:
                            symbol = extract_ticker(msg.text)
                            status = "NEW" if is_new else "SAME"

                        # Use safe_print for anything that may contain emoji
                        preview = msg.text[:150] + ("..." if len(msg.text) > 150 else "")
                        safe_print(f"\n[{msg.date}]")
                        safe_print(f"   Message : {preview}")
                        safe_print(f"   Ticker  : {symbol or '(not found)'}")
                        safe_print(f"   Status  : {'NEW' if is_new else 'Same as last poll'}")
                        safe_print("-" * 50)

                        if is_new and symbol and is_new_signal(status):
                            log.info(f"Dispatching buy signal for {symbol}")
                            threading.Thread(
                                target=process_buy_signal,
                                args=(symbol, ibkr),
                                daemon=True,
                            ).start()
                        elif is_new and symbol:
                            log.info(f"{symbol}: status='{status}' -- not NEW, skipping.")

                        last_seen_id = msg.id

                log.info(f"Waiting {POLL_INTERVAL}s...")
                await asyncio.sleep(POLL_INTERVAL)

        except FloodWaitError as e:
            log.warning(f"Rate-limited -- waiting {e.seconds}s")
            await asyncio.sleep(e.seconds + 5)
        except KeyboardInterrupt:
            log.info("Stopped by user.")
            break
        except Exception as e:
            log.error(f"Telegram error: {type(e).__name__}: {e}  -- reconnecting in 15s")
            await asyncio.sleep(15)
        finally:
            try:
                await client.disconnect()
            except Exception:
                pass


# ══════════════════════════════════════════════════════════════════════════════
# Entry Point
# ══════════════════════════════════════════════════════════════════════════════
def main():
    log.info("=== Buy Bot Starting ===")
    ibkr = IBKRBuyClient()
    ibkr.connect()
    try:
        asyncio.run(poll_telegram(ibkr))
    except KeyboardInterrupt:
        log.info("Buy bot stopped.")
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()

In [ ]:
"""
Buy Bot — Telegram Poller -> IBKR Paper Trading -> MongoDB

HOW THE EVENT LOOP WORKS (ib_insync 0.9.x):
  - util.startLoop() calls nest_asyncio.apply() which PATCHES the current
    event loop to allow nested awaiting — there is NO separate IB thread.
  - Both Telethon and ib_insync share the SAME asyncio event loop.
  - Because of nest_asyncio, ib_insync's sync methods (reqHistoricalData etc.)
    work fine when called from WITHIN an already-running loop.

ROOT CAUSE OF PREVIOUS DEADLOCK:
  - process_buy_signal() ran in a plain threading.Thread.
  - That thread has NO event loop, so ib_insync's sync wrappers (which call
    loop.run_until_complete internally) raised "no running event loop" or hung.

FIX:
  - Replace threading.Thread with asyncio.ensure_future so buy logic runs
    as a coroutine on the shared event loop.
  - All IB calls use the standard async variants (reqHistoricalDataAsync etc.)
    and are awaited normally — no threads, no run_coroutine_threadsafe.
"""

import asyncio
import os
import sys
import re
from datetime import datetime, timezone
import random
import logging

import nest_asyncio
nest_asyncio.apply()   # must be before any asyncio.run() / IB usage

import pandas as pd
from pymongo import MongoClient

from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import FloodWaitError, UserAlreadyParticipantError
from telethon.tl.functions.messages import ImportChatInviteRequest

from ib_insync import IB, Stock, LimitOrder, util

import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# Logging — UTF-8 safe (Jupyter + Windows cp1252)
# ══════════════════════════════════════════════════════════════════════════════
class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh  = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh  = logging.FileHandler("buy_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("buy_bot")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# Safe print — handles emoji in raw Telegram message text
# ══════════════════════════════════════════════════════════════════════════════
def safe_print(text: str):
    try:
        encoded = text.encode(sys.stdout.encoding or "utf-8", errors="replace")
        sys.stdout.buffer.write(encoded + b"\n")
        sys.stdout.buffer.flush()
    except AttributeError:
        print(text.encode("utf-8", errors="replace").decode("utf-8", errors="replace"))
    except Exception:
        print(text)


# ══════════════════════════════════════════════════════════════════════════════
# Config
# ══════════════════════════════════════════════════════════════════════════════
API_ID        = 34025130
API_HASH      = "e5e514ebcb8a15042feadfecd7a15cfe"
PHONE         = "+18477675748"
INVITE_LINK   = "https://t.me/+M5HRCxZycqJmOTg5"
POLL_INTERVAL = 20

SESSION_FILE   = "buy_bot_session.txt"
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)
ORDER_SIZE     = 10

mongo_client              = MongoClient("mongodb://localhost:27017/")
db                        = mongo_client["brkout_database"]
trades_collection         = db["trades"]
failed_entries_collection = db["failed_bad_entries"]
ticker_collection         = db["tickers"]
ticker_trade_collection   = db["tickers_trade"]


# ══════════════════════════════════════════════════════════════════════════════
# Telegram Parsers
# ══════════════════════════════════════════════════════════════════════════════
def extract_ticker(message_text: str) -> str | None:
    text = message_text.strip()
    m = re.match(r'^\[.*?\]\s+([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    m = re.match(r'^([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    return None


def parse_structured_message(text: str) -> dict | None:
    result = {"symbol": None, "status": None}
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^ticker\s*:", line):
            result["symbol"] = line.split(":", 1)[1].strip().upper()
        if re.match(r"(?i)^status\s*:", line):
            result["status"] = line.split(":", 1)[1].strip()
    if result["symbol"] and result["status"]:
        return result
    return None


def is_new_signal(status_text: str) -> bool:
    return "new" in status_text.lower()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR Client — fully async, runs on shared event loop
# ══════════════════════════════════════════════════════════════════════════════
class IBKRBuyClient:
    def __init__(self):
        self.ib = IB()

    def connect(self):
        # startLoop() applies nest_asyncio so sync IB calls work inside a
        # running loop. connect() itself is a sync blocking call.
        util.startLoop()
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── sync helpers (safe to call from async context via nest_asyncio) ───────
    def has_open_position(self, symbol: str) -> bool:
        positions = self.ib.positions()
        result = any(p.contract.symbol == symbol and p.position != 0 for p in positions)
        log.debug(f"has_open_position({symbol}) => {result}  (total={len(positions)})")
        return result

    def has_pending_order(self, symbol: str) -> bool:
        open_trades = self.ib.openTrades()
        result = any(t.contract.symbol == symbol for t in open_trades)
        log.debug(f"has_pending_order({symbol}) => {result}  (total={len(open_trades)})")
        return result

    # ── async market data ─────────────────────────────────────────────────────
    async def get_ask_price(self, symbol: str) -> float | None:
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        await asyncio.sleep(3)
        self.ib.cancelMktData(contract)
        log.debug(f"{symbol}: ask={ticker.ask}  last={ticker.last}  "
                  f"close={ticker.close}  bid={ticker.bid}")
        for label, val in [("ask", ticker.ask), ("last", ticker.last), ("close", ticker.close)]:
            if val and float(val) > 0:
                log.info(f"{symbol}: price source='{label}'  value=${float(val):.4f}")
                return float(val)
        log.warning(f"{symbol}: all price fields empty from IBKR")
        return None

    async def get_bars(self, symbol: str, n: int = 70) -> pd.DataFrame | None:
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        for what in ("TRADES", "MIDPOINT"):
            log.debug(f"{symbol}: requesting bars whatToShow={what} useRTH=False")
            raw = await self.ib.reqHistoricalDataAsync(
                contract,
                endDateTime="",
                durationStr="5 D",
                barSizeSetting="5 mins",
                whatToShow=what,
                useRTH=False,
            )
            log.debug(f"{symbol}: bars returned={len(raw) if raw else 0}")
            if raw:
                df = pd.DataFrame([
                    {"high": b.high, "low": b.low, "close": b.close}
                    for b in raw
                ]).tail(n).reset_index(drop=True)
                log.debug(f"{symbol}: df shape={df.shape}  last_close={df['close'].iloc[-1]:.4f}")
                return df
        log.warning(f"{symbol}: no bars returned from IBKR")
        return None

    async def ema_bullish(self, symbol: str) -> bool:
        log.debug(f"ema_bullish({symbol}): fetching bars...")
        try:
            df = await self.get_bars(symbol, n=70)
            if df is None:
                log.warning(f"{symbol}: no bars -- skipping EMA")
                return False
            if len(df) < 50:
                log.warning(f"{symbol}: only {len(df)} bars (need 50) -- skipping EMA")
                return False
            ema21 = df["close"].ewm(span=21, adjust=False).mean().iloc[-1]
            ema50 = df["close"].ewm(span=50, adjust=False).mean().iloc[-1]
            bullish = bool(ema21 > ema50)
            log.info(f"{symbol}: EMA21={ema21:.4f}  EMA50={ema50:.4f}  bullish={bullish}")
            return bullish
        except Exception as e:
            log.error(f"{symbol}: EMA error - {e}")
            return False

    async def place_limit_buy(self, symbol: str, ask: float, qty: int = ORDER_SIZE):
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        order = LimitOrder("BUY", qty, round(ask, 2))
        order.outsideRth = True   # allows pre/post market execution
        trade = self.ib.placeOrder(contract, order)
        await asyncio.sleep(1)
        log.info(f"BUY {symbol} x{qty} @ ${ask:.2f}  "
                 f"orderId={trade.order.orderId}  outsideRth=True  "
                 f"status={trade.orderStatus.status}")
        return trade

    async def get_avg_fill_price(self, symbol: str, order_id: int) -> float | None:
        await asyncio.sleep(2)
        for trade in self.ib.trades():
            if (trade.contract.symbol == symbol
                    and trade.order.orderId == order_id
                    and trade.orderStatus.avgFillPrice > 0):
                avg = float(trade.orderStatus.avgFillPrice)
                log.info(f"{symbol}: avg fill from broker = ${avg:.4f}")
                return avg
        fills = self.ib.executions()
        sf = [f for f in fills
              if f.contract.symbol == symbol and f.execution.orderId == order_id]
        if sf:
            total_qty  = sum(f.execution.shares for f in sf)
            total_cost = sum(f.execution.shares * f.execution.price for f in sf)
            avg = total_cost / total_qty
            log.info(f"{symbol}: avg fill from executions = ${avg:.4f}")
            return avg
        log.warning(f"{symbol}: avg fill not available for orderId={order_id}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# MongoDB helpers
# ══════════════════════════════════════════════════════════════════════════════
def save_trade(symbol: str, limit_price: float, avg_fill_price: float | None,
               qty: int, order_id: int):
    entry_price = avg_fill_price if avg_fill_price else limit_price
    doc = {
        "symbol":         symbol,
        "limit_price":    limit_price,
        "entry_price":    entry_price,
        "avg_fill_price": avg_fill_price,
        "qty":            qty,
        "order_id":       order_id,
        "status":         "open",
        "entered_at":     datetime.now(timezone.utc),
        "exit_price":     None,
        "exit_reason":    None,
        "exited_at":      None,
        "pnl_pct":        None,
    }
    trades_collection.insert_one(doc)
    ticker_trade_collection.update_one(
        {"symbol": symbol},
        {"$set": {"last_trade": doc, "updated_at": datetime.now(timezone.utc)}},
        upsert=True,
    )
    fill_str = f"${avg_fill_price:.4f}" if avg_fill_price else "pending"
    log.info(f"trade saved  {symbol}  limit=${limit_price:.2f}  avg_fill={fill_str}")


def save_failed_entry(symbol: str, reason: str):
    failed_entries_collection.insert_one({
        "symbol":    symbol,
        "reason":    reason,
        "timestamp": datetime.now(timezone.utc),
    })
    log.info(f"failed entry {symbol} -- {reason}")


# ══════════════════════════════════════════════════════════════════════════════
# Buy Logic — async coroutine (runs on shared event loop, no threads needed)
# ══════════════════════════════════════════════════════════════════════════════
async def process_buy_signal(symbol: str, ibkr: IBKRBuyClient):
    log.info(f"========== BUY SIGNAL: {symbol} ==========")

    # Gate 1 & 2: position / pending order (sync, safe with nest_asyncio)
    if ibkr.has_open_position(symbol):
        save_failed_entry(symbol, "already_in_position"); return
    if ibkr.has_pending_order(symbol):
        save_failed_entry(symbol, "pending_order_exists"); return

    # Gate 3: EMA check (async)
    log.debug(f"{symbol}: GATE 3 - EMA check...")
    if not await ibkr.ema_bullish(symbol):
        save_failed_entry(symbol, "ema_not_bullish"); return
    log.debug(f"{symbol}: GATE 3 PASSED")

    # Gate 4: price (async)
    log.debug(f"{symbol}: GATE 4 - ask price...")
    ask = await ibkr.get_ask_price(symbol)
    if not ask:
        save_failed_entry(symbol, "no_ask_price"); return
    log.debug(f"{symbol}: GATE 4 PASSED  ask=${ask:.4f}")

    # Place order (async)
    trade    = await ibkr.place_limit_buy(symbol, ask, ORDER_SIZE)
    order_id = trade.order.orderId
    avg_fill = await ibkr.get_avg_fill_price(symbol, order_id)

    save_trade(symbol, ask, avg_fill, ORDER_SIZE, order_id)
    ticker_collection.update_one(
        {"symbol": symbol},
        {"$set": {"last_signal": datetime.now(timezone.utc), "status": "bought"}},
        upsert=True,
    )
    log.info(f"========== BUY COMPLETE: {symbol} @ ${ask:.4f} ==========")


# ══════════════════════════════════════════════════════════════════════════════
# Session helpers
# ══════════════════════════════════════════════════════════════════════════════
def load_session_string() -> str:
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, encoding="utf-8") as f:
            return f.read().strip()
    return ""


def save_session_string(session_str: str):
    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(session_str)
    log.info(f"Session saved -> {SESSION_FILE}")


# ══════════════════════════════════════════════════════════════════════════════
# Telegram Poller
# ══════════════════════════════════════════════════════════════════════════════
async def poll_telegram(ibkr: IBKRBuyClient):
    while True:
        session_str = load_session_string()
        client = TelegramClient(StringSession(session_str), API_ID, API_HASH)
        try:
            await client.start(phone=PHONE)
            save_session_string(client.session.save())

            m = re.search(r't\.me/(?:joinchat/|\+)([A-Za-z0-9_-]+)', INVITE_LINK)
            if not m:
                raise ValueError(f"Cannot parse invite hash: {INVITE_LINK}")
            try:
                await client(ImportChatInviteRequest(m.group(1)))
                log.info("Joined channel")
            except UserAlreadyParticipantError:
                log.info("Already a member")

            channel = await client.get_entity(INVITE_LINK)
            log.info(f"Channel: {channel.title}  (ID: {channel.id})")
            safe_print("-" * 60)

            last_seen_id = None

            while True:
                messages = await client.get_messages(channel, limit=1)
                if messages:
                    msg = messages[0]
                    if msg.text:
                        is_new = (last_seen_id is None or msg.id != last_seen_id)

                        parsed = parse_structured_message(msg.text)
                        if parsed:
                            symbol = parsed["symbol"]
                            status = parsed["status"]
                        else:
                            symbol = extract_ticker(msg.text)
                            status = "NEW" if is_new else "SAME"

                        preview = msg.text[:150] + ("..." if len(msg.text) > 150 else "")
                        safe_print(f"\n[{msg.date}]")
                        safe_print(f"   Message : {preview}")
                        safe_print(f"   Ticker  : {symbol or '(not found)'}")
                        safe_print(f"   Status  : {'NEW' if is_new else 'Same as last poll'}")
                        log.debug(f"is_new={is_new}  symbol={symbol}  status='{status}'")
                        safe_print("-" * 60)

                        if is_new and symbol and is_new_signal(status):
                            log.info(f"Dispatching buy coroutine for {symbol}")
                            # Schedule as a fire-and-forget task on the shared loop
                            asyncio.ensure_future(process_buy_signal(symbol, ibkr))
                        elif is_new and symbol:
                            log.info(f"{symbol}: status='{status}' -- not NEW, skipping.")

                        last_seen_id = msg.id

                log.info(f"Waiting {POLL_INTERVAL}s...")
                await asyncio.sleep(POLL_INTERVAL)

        except FloodWaitError as e:
            log.warning(f"Rate-limited -- waiting {e.seconds}s")
            await asyncio.sleep(e.seconds + 5)
        except KeyboardInterrupt:
            log.info("Stopped by user.")
            break
        except Exception as e:
            log.error(f"Telegram error: {type(e).__name__}: {e}  -- reconnecting in 15s")
            await asyncio.sleep(15)
        finally:
            try:
                await client.disconnect()
            except Exception:
                pass


# ══════════════════════════════════════════════════════════════════════════════
# Entry Point
# ══════════════════════════════════════════════════════════════════════════════
def main():
    log.info("=== Buy Bot Starting ===")
    ibkr = IBKRBuyClient()
    ibkr.connect()
    try:
        asyncio.run(poll_telegram(ibkr))
    except KeyboardInterrupt:
        log.info("Buy bot stopped.")
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()


In [ ]:
"""
Unified Trading Bot
═══════════════════
Telegram Poller  ──►  MongoDB (watchlist_tickers)  ──►  IBKR Trading Loop

Flow:
  1. Telethon polls a private Telegram channel every 20 seconds.
  2. NEW signals are parsed for a ticker symbol and upserted into
     the 'watchlist_tickers' MongoDB collection.
  3. The IBKR trading loop runs every 5 minutes, reads ALL active
     tickers from 'watchlist_tickers', qualifies contracts on-the-fly,
     and evaluates buy / position-management logic for each.
  4. At EOD (>= 4 PM) the watchlist_tickers collection is cleared.

EVENT LOOP:
  Both Telethon and ib_insync share the SAME asyncio event loop via
  nest_asyncio. The Telegram poller and the trading loop run as
  concurrent asyncio tasks — no threads needed.
"""

import asyncio
import os
import re
import sys
import random
import logging
import warnings
from datetime import datetime, timezone, timedelta

import nest_asyncio
nest_asyncio.apply()          # must be before any IB / asyncio.run() usage

import numpy as np
import pandas as pd
from pymongo import MongoClient
from bson import ObjectId

from ib_insync import IB, Stock, MarketOrder, util

from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import FloodWaitError, UserAlreadyParticipantError
from telethon.tl.functions.messages import ImportChatInviteRequest

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt    = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh     = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh     = logging.FileHandler("unified_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("unified_bot")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


def safe_print(text: str):
    try:
        encoded = text.encode(sys.stdout.encoding or "utf-8", errors="replace")
        sys.stdout.buffer.write(encoded + b"\n")
        sys.stdout.buffer.flush()
    except Exception:
        print(text)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# Telegram
API_ID        = 34025130
API_HASH      = "e5e514ebcb8a15042feadfecd7a15cfe"
PHONE         = "+18477675748"
INVITE_LINK   = "https://t.me/+M5HRCxZycqJmOTg5"
POLL_INTERVAL = 20          # seconds between Telegram polls
SESSION_FILE  = "unified_bot_session.txt"

# IBKR
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)
ORDER_SIZE     = 20

# Trading parameters
TREND_SCORE_ENTRY    = 85     # minimum trend score to enter
TREND_SCORE_WATCHLIST = 85    # minimum to remain on watchlist
TREND_SCORE_DROP_EXIT = 69    # trend score below this triggers soft stop
STOP_LOSS_1          = 0.10   # 10% soft stop (checks trend score)
STOP_LOSS_2          = 0.15   # 15% hard stop
TAKE_PROFIT_PCT      = 0.50   # 50% take profit
BASE_TRAILING_AMT    = 1.0    # $1 fixed trailing floor
MIN_TRAILING_PCT     = 0.005  # 0.5% trailing floor
ATR_MULTIPLIER       = 1.5
EOD_CLEAR_HOUR       = 16     # 4 PM — clear watchlist collection


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB
# ══════════════════════════════════════════════════════════════════════════════

mongo_client = MongoClient("mongodb://localhost:27017/")
db           = mongo_client["trading_db"]

watchlist_col  = db["watchlist_tickers"]    # ← shared ticker list (Telegram → Trading loop)
trades_col     = db["trades_v2"]
excluded_col   = db["excluded_tickers_v2"]

# Ensure unique index on 'symbol' so duplicate inserts are a no-op
watchlist_col.create_index("symbol", unique=True)


# ── Watchlist helpers ─────────────────────────────────────────────────────────

def watchlist_add(symbol: str, source: str = "telegram") -> bool:
    """
    Insert ticker into watchlist_tickers.
    Returns True if inserted, False if already existed (silently ignored).
    """
    try:
        watchlist_col.insert_one({
            "symbol":     symbol,
            "source":     source,
            "added_at":   datetime.now(timezone.utc),
            "active":     True,
        })
        log.info(f"WATCHLIST ADD  : {symbol}  (source={source})")
        return True
    except Exception:
        # Duplicate key — already in collection, skip silently
        log.debug(f"WATCHLIST DUP  : {symbol} already present, skipped.")
        return False


def watchlist_get_all() -> list[str]:
    """Return all active ticker symbols currently in the watchlist."""
    docs = watchlist_col.find({"active": True}, {"symbol": 1})
    return [d["symbol"] for d in docs]


def watchlist_clear_eod():
    """Delete ALL documents from watchlist_tickers at/after EOD hour."""
    now = datetime.now()
    if now.hour >= EOD_CLEAR_HOUR:
        result = watchlist_col.delete_many({})
        if result.deleted_count:
            log.info(f"EOD CLEAR: removed {result.deleted_count} ticker(s) from watchlist_tickers")


def watchlist_print():
    symbols = watchlist_get_all()
    if symbols:
        log.info(f"Watchlist ({len(symbols)}): {', '.join(symbols)}")
    else:
        log.info("Watchlist: empty")


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM PARSERS
# ══════════════════════════════════════════════════════════════════════════════

def extract_ticker(text: str) -> str | None:
    text = text.strip()
    m = re.match(r'^\[.*?\]\s+([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    m = re.match(r'^([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    return None


def parse_structured_message(text: str) -> dict | None:
    result = {"symbol": None, "status": None}
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^ticker\s*:", line):
            result["symbol"] = line.split(":", 1)[1].strip().upper()
        if re.match(r"(?i)^status\s*:", line):
            result["status"] = line.split(":", 1)[1].strip()
    if result["symbol"] and result["status"]:
        return result
    return None


def is_new_signal(status_text: str) -> bool:
    return "new" in status_text.lower()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

ib = IB()

def ibkr_connect():
    util.startLoop()
    ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
    log.info(f"IBKR connected | accounts: {ib.wrapper.accounts}")


# ── Gate helpers ──────────────────────────────────────────────────────────────

def has_open_position(symbol: str) -> bool:
    """Gate 1 — existing IB position check."""
    result = any(p.contract.symbol == symbol and p.position != 0 for p in ib.positions())
    log.debug(f"Gate 1 | has_open_position({symbol}) => {result}")
    return result


def has_pending_order(symbol: str) -> bool:
    """Gate 2 — pending IB order check."""
    result = any(t.contract.symbol == symbol for t in ib.openTrades())
    log.debug(f"Gate 2 | has_pending_order({symbol}) => {result}")
    return result


def get_ask_price(contract) -> float | None:
    """Gate 4 — fetch live ask / last / close from IB market data."""
    ticker = ib.reqMktData(contract, "106", False, False)
    ib.sleep(3)
    ib.cancelMktData(contract)
    for label, val in [("ask", ticker.ask), ("last", ticker.last), ("close", ticker.close)]:
        if val and float(val) > 0:
            log.info(f"Gate 4 | {contract.symbol}: source='{label}'  ${float(val):.4f}")
            return float(val)
    log.warning(f"Gate 4 | {contract.symbol}: all price fields empty")
    return None


# ══════════════════════════════════════════════════════════════════════════════
# INDICATOR CALCULATIONS
# ══════════════════════════════════════════════════════════════════════════════

def calculate_ema(df: pd.DataFrame, period: int) -> pd.Series:
    return df["close"].ewm(span=period, adjust=False).mean()


def calculate_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    high_low   = df["high"] - df["low"]
    high_close = np.abs(df["high"] - df["close"].shift())
    low_close  = np.abs(df["low"]  - df["close"].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["ATR"]  = true_range.rolling(period).mean()
    return df


def calculate_ema_200(df: pd.DataFrame) -> pd.DataFrame:
    df["ema_200"] = calculate_ema(df, 200)
    return df


def calculate_session_vwap(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date_only"] = pd.to_datetime(df["date"]).dt.date
    today           = df["date_only"].max()
    mask            = df["date_only"] == today
    today_df        = df.loc[mask].copy()
    today_df["vwap"] = (
        (today_df["close"] * today_df["volume"]).cumsum() /
        today_df["volume"].cumsum()
    )
    df.loc[mask, "vwap"] = today_df["vwap"].values
    df["vwap"] = df["vwap"].ffill()
    return df


def calculate_ema_alignment_oscillator(df: pd.DataFrame,
                                       fast_period=8,
                                       mid_period=13,
                                       slow_period=14,
                                       trend_period=21,
                                       slope_threshold=1.0,
                                       spacing_min=0.2) -> pd.DataFrame:
    df["ema_fast"]  = calculate_ema(df, fast_period)
    df["ema_mid"]   = calculate_ema(df, mid_period)
    df["ema_slow"]  = calculate_ema(df, slow_period)
    df["ema_trend"] = calculate_ema(df, trend_period)
    df = calculate_atr(df, 14)

    def slope_deg(ema_s, atr_s):
        s = (ema_s - ema_s.shift(1)) / (atr_s + 0.0001) * 100
        return np.degrees(np.arctan(s))

    df["fast_angle"] = slope_deg(df["ema_fast"], df["ATR"])
    df["mid_angle"]  = slope_deg(df["ema_mid"],  df["ATR"])
    df["slow_angle"] = slope_deg(df["ema_slow"], df["ATR"])

    df["bullish_stack"] = (
        (df["ema_fast"]  > df["ema_mid"]) &
        (df["ema_mid"]   > df["ema_slow"]) &
        (df["ema_slow"]  > df["ema_trend"])
    )
    df["bearish_stack"] = (
        (df["ema_fast"]  < df["ema_mid"]) &
        (df["ema_mid"]   < df["ema_slow"]) &
        (df["ema_slow"]  < df["ema_trend"])
    )

    df["price_above_fast"] = df["close"] > df["ema_fast"]
    df["price_below_fast"] = df["close"] < df["ema_fast"]

    df["spacing_fast_mid"] = np.abs(df["ema_fast"] - df["ema_mid"]) / df["ema_mid"] * 100
    df["spacing_mid_slow"] = np.abs(df["ema_mid"]  - df["ema_slow"]) / df["ema_slow"] * 100
    df["avg_spacing"]      = (df["spacing_fast_mid"] + df["spacing_mid_slow"]) / 2

    df["volume_sma"]          = df["volume"].rolling(20).mean()
    df["volume_confirmation"] = df["volume"] > df["volume_sma"]

    df["trend_score"] = 0.0

    for i in range(1, len(df)):
        prev = df["trend_score"].iloc[i - 1]

        if df["bullish_stack"].iloc[i] and df["price_above_fast"].iloc[i]:
            avg_slope     = (df["fast_angle"].iloc[i] + df["mid_angle"].iloc[i] + df["slow_angle"].iloc[i]) / 3
            slope_bonus   = min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = 10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = 40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        elif df["bearish_stack"].iloc[i] and df["price_below_fast"].iloc[i]:
            avg_slope     = (abs(df["fast_angle"].iloc[i]) + abs(df["mid_angle"].iloc[i]) + abs(df["slow_angle"].iloc[i])) / 3
            slope_bonus   = -min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = -min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = -10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = -40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        else:
            df.loc[df.index[i], "trend_score"] = prev * 0.9

    df.loc[~df["volume_confirmation"], "trend_score"] *= 0.8
    df["trend_score"] = df["trend_score"].clip(-100, 100)

    # Threshold = 85
    df["strong_bull_signal"] = (
        (df["trend_score"] >= TREND_SCORE_ENTRY) &
        (df["trend_score"] > df["trend_score"].shift(1))
    )
    df["strong_bear_signal"] = (
        (df["trend_score"] <= -TREND_SCORE_ENTRY) &
        (df["trend_score"] < df["trend_score"].shift(1))
    )
    return df


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENT HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_document(symbol, entry_price, quantity, trend_score_at_entry):
    return {
        "instrument":        symbol,
        "direction":         "long",
        "entry_price":       entry_price,
        "highest_price":     entry_price,
        "quantity":          quantity,
        "entry_timestamp":   datetime.now(),
        "entry_trend_score": trend_score_at_entry,
        "order_id":          str(ObjectId()),
        "exit_price":        None,
        "exit_timestamp":    None,
        "reason":            None,
        "current_pnl":       0,
        "realized_pnl":      0,
        "status":            "open",
        "created_at":        datetime.now(),
        "updated_at":        datetime.now(),
    }


def update_trade_document(trade_id, updates: dict):
    updates["updated_at"] = datetime.now()
    trades_col.update_one({"_id": ObjectId(trade_id)}, {"$set": updates})


# ══════════════════════════════════════════════════════════════════════════════
# TRADING LOOP  (runs as asyncio task every 5 minutes)
# ══════════════════════════════════════════════════════════════════════════════

async def trading_loop():
    log.info("Trading loop started.")
    await asyncio.sleep(10)   # brief startup grace period

    while True:
        # ── EOD cleanup ───────────────────────────────────────────────────────
        watchlist_clear_eod()
        watchlist_print()

        # ── Load tickers dynamically from MongoDB ─────────────────────────────
        symbols = watchlist_get_all()

        if not symbols:
            log.info("Watchlist empty — nothing to evaluate. Waiting 5 min...")
            await asyncio.sleep(300)
            continue

        log.info(f"Evaluating {len(symbols)} ticker(s): {', '.join(symbols)}")

        # Qualify contracts fresh each cycle (watchlist may have changed)
        contracts = [Stock(s, "SMART", "USD") for s in symbols]
        try:
            ib.qualifyContracts(*contracts)
        except Exception as e:
            log.warning(f"qualifyContracts error: {e}")

        for contract in contracts:
            symbol = contract.symbol
            log.info(f"\n{'='*60}")
            log.info(f"Evaluating {symbol} at {datetime.now().strftime('%H:%M:%S')}")
            log.info(f"{'='*60}")

            # ── Fetch 5-min bars ──────────────────────────────────────────────
            try:
                bars = ib.reqHistoricalData(
                    contract,
                    endDateTime="",
                    durationStr="2 D",
                    barSizeSetting="5 mins",
                    whatToShow="TRADES",
                    useRTH=False,
                    formatDate=1,
                )
            except Exception as e:
                log.warning(f"{symbol}: reqHistoricalData error — {e}")
                continue

            if not bars:
                log.warning(f"{symbol}: no historical data returned")
                continue

            df = util.df(bars)
            df.columns = [c.lower() for c in df.columns]

            # ── Calculate indicators ──────────────────────────────────────────
            df = calculate_ema_alignment_oscillator(df)
            df = calculate_ema_200(df)
            df = calculate_session_vwap(df)

            row           = df.iloc[-1]
            current_price = row["close"]
            trend_score   = row["trend_score"]
            atr           = row.get("ATR", BASE_TRAILING_AMT)
            vwap          = row["vwap"]

            log.info(f"Price: ${current_price:.2f} | Score: {trend_score:.1f} | "
                     f"EMA200: ${row['ema_200']:.2f} | VWAP: ${vwap:.2f} | "
                     f"BullSig: {row['strong_bull_signal']}")

            # ── Check for open MongoDB trade ──────────────────────────────────
            open_trade = trades_col.find_one({"instrument": symbol, "status": "open"})

            # ═════════════════════════════════════════════════════════════════
            # SELL / POSITION MANAGEMENT
            # ═════════════════════════════════════════════════════════════════

            if open_trade:
                entry_price   = open_trade["entry_price"]
                highest_price = open_trade.get("highest_price", entry_price)
                quantity      = open_trade["quantity"]
                time_in_trade = (datetime.now() - open_trade["entry_timestamp"]).total_seconds() / 3600

                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade["_id"], {"highest_price": highest_price})

                profit_pct  = (current_price - entry_price) / entry_price
                pnl_ps      = current_price - entry_price
                current_pnl = pnl_ps * quantity
                update_trade_document(open_trade["_id"], {"current_pnl": current_pnl})

                log.info(f"OPEN | Entry: ${entry_price:.2f} | P&L: {profit_pct:.2%} | Score: {trend_score:.1f}")

                # ── Stop loss ─────────────────────────────────────────────────
                should_exit = False
                exit_reason = None

                if profit_pct <= -STOP_LOSS_1:
                    if trend_score < TREND_SCORE_DROP_EXIT:
                        should_exit = True
                        exit_reason = "stop_loss_10pct_trend_weak"
                        log.info(f"EXIT: 10% loss + score {trend_score:.1f} < {TREND_SCORE_DROP_EXIT}")
                    elif profit_pct <= -STOP_LOSS_2:
                        should_exit = True
                        exit_reason = "stop_loss_15pct_hard"
                        log.info(f"EXIT: 15% hard stop reached")
                    else:
                        log.info(f"Holding at {profit_pct:.2%} loss — trend still strong ({trend_score:.1f})")

                if should_exit and quantity > 0:
                    ib.placeOrder(contract, MarketOrder("SELL", quantity))
                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           exit_reason,
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"STOP LOSS | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Take profit ───────────────────────────────────────────────
                if profit_pct >= TAKE_PROFIT_PCT:
                    ib.placeOrder(contract, MarketOrder("SELL", quantity))
                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           f"take_profit_{profit_pct:.2%}",
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"TAKE PROFIT | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Trailing stop (only when profitable) ──────────────────────
                if profit_pct > 0:
                    atr_stop  = highest_price - (atr * ATR_MULTIPLIER)
                    pct_stop  = highest_price * (1 - MIN_TRAILING_PCT)
                    fix_stop  = highest_price - BASE_TRAILING_AMT
                    t_factor  = min(1.0, time_in_trade / 24)
                    trail_stop = max(atr_stop * (1 - 0.25 * t_factor), pct_stop, fix_stop)

                    if current_price <= trail_stop and quantity > 0:
                        ib.placeOrder(contract, MarketOrder("SELL", quantity))
                        realized = pnl_ps * quantity
                        update_trade_document(open_trade["_id"], {
                            "exit_price":       current_price,
                            "exit_timestamp":   datetime.now(),
                            "reason":           "trailing_stop_hit",
                            "realized_pnl":     realized,
                            "exit_trend_score": trend_score,
                            "status":           "closed",
                        })
                        log.info(f"TRAILING STOP | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                        await asyncio.sleep(0.5)
                        continue

            # ═════════════════════════════════════════════════════════════════
            # BUY / ENTRY LOGIC
            # ═════════════════════════════════════════════════════════════════

            else:
                # Already traded today?
                today_str     = datetime.now().date().isoformat()
                exclude_entry = excluded_col.find_one({"ticker": symbol, "date": today_str})
                if exclude_entry:
                    log.info(f"{symbol}: already traded today — skipping")
                    await asyncio.sleep(0.5)
                    continue

                # ── Entry condition checks ────────────────────────────────────
                above_ema200 = current_price > row["ema_200"]
                above_vwap   = current_price > vwap
                bull_signal  = bool(row["strong_bull_signal"])

                if not bull_signal:
                    log.info(f"{symbol}: no signal (score={trend_score:.1f}, need >={TREND_SCORE_ENTRY} & rising)")
                    await asyncio.sleep(0.5)
                    continue
                if not above_ema200:
                    log.info(f"{symbol}: below EMA200 (${current_price:.2f} < ${row['ema_200']:.2f})")
                    await asyncio.sleep(0.5)
                    continue
                if not above_vwap:
                    log.info(f"{symbol}: below VWAP (${current_price:.2f} < ${vwap:.2f})")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 1: no existing IB position ──────────────────────────
                if has_open_position(symbol):
                    log.info(f"{symbol}: Gate 1 FAILED — existing IB position")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 2: no pending IB order ───────────────────────────────
                if has_pending_order(symbol):
                    log.info(f"{symbol}: Gate 2 FAILED — pending IB order")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 4: valid ask price ───────────────────────────────────
                ask_price = get_ask_price(contract)
                if not ask_price:
                    log.info(f"{symbol}: Gate 4 FAILED — no valid ask price")
                    await asyncio.sleep(0.5)
                    continue

                # ── All gates passed → place order ────────────────────────────
                ib.placeOrder(contract, MarketOrder("BUY", ORDER_SIZE))

                trade_doc = create_trade_document(
                    symbol               = symbol,
                    entry_price          = ask_price,
                    quantity             = ORDER_SIZE,
                    trend_score_at_entry = trend_score,
                )
                trades_col.insert_one(trade_doc)

                excluded_col.insert_one({"ticker": symbol, "date": today_str})

                log.info(
                    f"BUY | {symbol} | ask=${ask_price:.2f} | qty={ORDER_SIZE} | "
                    f"score={trend_score:.1f} | EMA200 ✓ | VWAP ✓ | G1 ✓ | G2 ✓ | G4 ✓"
                )

            await asyncio.sleep(0.5)   # brief pause between tickers

        # ── End of scan cycle ─────────────────────────────────────────────────
        log.info(f"\n{'='*60}")
        log.info("Scan complete. Next scan in 5 minutes.")
        watchlist_print()
        log.info(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM POLLER  (runs as asyncio task continuously)
# Adds new tickers to MongoDB watchlist_tickers collection.
# ══════════════════════════════════════════════════════════════════════════════

def load_session() -> str:
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, encoding="utf-8") as f:
            return f.read().strip()
    return ""


def save_session(s: str):
    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(s)
    log.info(f"Telegram session saved → {SESSION_FILE}")


async def telegram_poller():
    log.info("Telegram poller started.")

    while True:
        tg = TelegramClient(StringSession(load_session()), API_ID, API_HASH)
        try:
            await tg.start(phone=PHONE)
            save_session(tg.session.save())

            # Join channel if needed
            m = re.search(r't\.me/(?:joinchat/|\+)([A-Za-z0-9_-]+)', INVITE_LINK)
            if not m:
                raise ValueError(f"Cannot parse invite hash: {INVITE_LINK}")
            try:
                await tg(ImportChatInviteRequest(m.group(1)))
                log.info("Joined Telegram channel")
            except UserAlreadyParticipantError:
                log.info("Already a member of the channel")

            channel      = await tg.get_entity(INVITE_LINK)
            last_seen_id = None
            log.info(f"Polling channel: {channel.title} (ID: {channel.id})")

            while True:
                messages = await tg.get_messages(channel, limit=1)
                if messages:
                    msg    = messages[0]
                    is_new = (last_seen_id is None or msg.id != last_seen_id)

                    if msg.text and is_new:
                        parsed = parse_structured_message(msg.text)
                        if parsed:
                            symbol = parsed["symbol"]
                            status = parsed["status"]
                        else:
                            symbol = extract_ticker(msg.text)
                            status = "NEW"

                        preview = msg.text[:120] + ("..." if len(msg.text) > 120 else "")
                        safe_print(f"\n[{msg.date}]")
                        safe_print(f"   Message : {preview}")
                        safe_print(f"   Ticker  : {symbol or '(not found)'}")
                        safe_print(f"   Status  : NEW")

                        if symbol and is_new_signal(status):
                            inserted = watchlist_add(symbol, source="telegram")
                            if inserted:
                                safe_print(f"   → Added {symbol} to watchlist_tickers in MongoDB")
                            else:
                                safe_print(f"   → {symbol} already in watchlist_tickers, skipped")
                        elif symbol:
                            log.info(f"{symbol}: status='{status}' — not NEW, skipping watchlist add")

                        last_seen_id = msg.id

                log.debug(f"Telegram: waiting {POLL_INTERVAL}s...")
                await asyncio.sleep(POLL_INTERVAL)

        except FloodWaitError as e:
            log.warning(f"Telegram rate-limited — waiting {e.seconds}s")
            await asyncio.sleep(e.seconds + 5)
        except KeyboardInterrupt:
            log.info("Telegram poller stopped by user.")
            break
        except Exception as e:
            log.error(f"Telegram error: {type(e).__name__}: {e} — reconnecting in 15s")
            await asyncio.sleep(15)
        finally:
            try:
                await tg.disconnect()
            except Exception:
                pass


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT — run both tasks concurrently on the same event loop
# ══════════════════════════════════════════════════════════════════════════════

async def main():
    log.info("=== Unified Trading Bot Starting ===")
    ibkr_connect()
    await asyncio.gather(
        telegram_poller(),   # continuously polls Telegram & updates watchlist_tickers
        trading_loop(),      # every 5 min reads watchlist_tickers & manages trades
    )


if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        log.info("Bot stopped by user.")
    finally:
        ib.disconnect()
        log.info("IBKR disconnected.")


In [ ]:
"""
Unified Trading Bot
═══════════════════
Telegram Poller  ──►  MongoDB (watchlist_tickers)  ──►  IBKR Trading Loop

Flow:
  1. Telethon polls a private Telegram channel every 20 seconds.
  2. NEW signals are parsed for a ticker symbol and upserted into
     the 'watchlist_tickers' MongoDB collection.
  3. The IBKR trading loop runs every 5 minutes, reads ALL active
     tickers from 'watchlist_tickers', qualifies contracts on-the-fly,
     and evaluates buy / position-management logic for each.
  4. At EOD (>= 4 PM) the watchlist_tickers collection is cleared.

EVENT LOOP:
  Both Telethon and ib_insync share the SAME asyncio event loop via
  nest_asyncio. The Telegram poller and the trading loop run as
  concurrent asyncio tasks — no threads needed.
"""

import asyncio
import os
import re
import sys
import random
import logging
import warnings
from datetime import datetime, timezone, timedelta

import nest_asyncio
nest_asyncio.apply()          # must be before any IB / asyncio.run() usage

import numpy as np
import pandas as pd
from pymongo import MongoClient
from bson import ObjectId

from ib_insync import IB, Stock, MarketOrder, util

from telethon import TelegramClient
from telethon.sessions import StringSession
from telethon.errors import FloodWaitError, UserAlreadyParticipantError
from telethon.tl.functions.messages import ImportChatInviteRequest

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt    = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh     = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh     = logging.FileHandler("unified_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("unified_bot")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


def safe_print(text: str):
    try:
        encoded = text.encode(sys.stdout.encoding or "utf-8", errors="replace")
        sys.stdout.buffer.write(encoded + b"\n")
        sys.stdout.buffer.flush()
    except Exception:
        print(text)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

# Telegram
API_ID        = 34025130
API_HASH      = "e5e514ebcb8a15042feadfecd7a15cfe"
PHONE         = "+18477675748"
INVITE_LINK   = "https://t.me/+M5HRCxZycqJmOTg5"
POLL_INTERVAL = 20          # seconds between Telegram polls
SESSION_FILE  = "unified_bot_session.txt"

# IBKR
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)
ORDER_SIZE     = 20

# Trading parameters
TREND_SCORE_ENTRY    = 85     # minimum trend score to enter
TREND_SCORE_WATCHLIST = 85    # minimum to remain on watchlist
TREND_SCORE_DROP_EXIT = 69    # trend score below this triggers soft stop
STOP_LOSS_1          = 0.10   # 10% soft stop (checks trend score)
STOP_LOSS_2          = 0.15   # 15% hard stop
TAKE_PROFIT_PCT      = 0.50   # 50% take profit
BASE_TRAILING_AMT    = 1.0    # $1 fixed trailing floor
MIN_TRAILING_PCT     = 0.005  # 0.5% trailing floor
ATR_MULTIPLIER       = 1.5
EOD_CLEAR_HOUR       = 16     # 4 PM — clear watchlist collection


# ══════════════════════════════════════════════════════════════════════════════
# MONGODB
# ══════════════════════════════════════════════════════════════════════════════

mongo_client = MongoClient("mongodb://localhost:27017/")
db           = mongo_client["breakout_db"]

watchlist_col  = db["watchlist_tickers"]    # ← shared ticker list (Telegram → Trading loop)
trades_col     = db["trades_v2"]
excluded_col   = db["excluded_tickers_v2"]

# Ensure unique index on 'symbol' so duplicate inserts are a no-op
watchlist_col.create_index("symbol", unique=True)


# ── Watchlist helpers ─────────────────────────────────────────────────────────

def watchlist_add(symbol: str, source: str = "telegram") -> bool:
    """
    Insert ticker into watchlist_tickers.
    Returns True if inserted, False if already existed (silently ignored).
    """
    try:
        watchlist_col.insert_one({
            "symbol":     symbol,
            "source":     source,
            "added_at":   datetime.now(timezone.utc),
            "active":     True,
        })
        log.info(f"WATCHLIST ADD  : {symbol}  (source={source})")
        return True
    except Exception:
        # Duplicate key — already in collection, skip silently
        log.debug(f"WATCHLIST DUP  : {symbol} already present, skipped.")
        return False


def watchlist_get_all() -> list[str]:
    """Return all active ticker symbols currently in the watchlist."""
    docs = watchlist_col.find({"active": True}, {"symbol": 1})
    return [d["symbol"] for d in docs]


def watchlist_clear_eod():
    """Delete ALL documents from watchlist_tickers at/after EOD hour."""
    now = datetime.now()
    if now.hour >= EOD_CLEAR_HOUR:
        result = watchlist_col.delete_many({})
        if result.deleted_count:
            log.info(f"EOD CLEAR: removed {result.deleted_count} ticker(s) from watchlist_tickers")


def watchlist_print():
    symbols = watchlist_get_all()
    if symbols:
        log.info(f"Watchlist ({len(symbols)}): {', '.join(symbols)}")
    else:
        log.info("Watchlist: empty")


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM PARSERS
# ══════════════════════════════════════════════════════════════════════════════

def extract_ticker(text: str) -> str | None:
    text = text.strip()
    m = re.match(r'^\[.*?\]\s+([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    m = re.match(r'^([A-Z]{1,5})\b', text)
    if m:
        return m.group(1)
    return None


def parse_structured_message(text: str) -> dict | None:
    result = {"symbol": None, "status": None}
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^ticker\s*:", line):
            result["symbol"] = line.split(":", 1)[1].strip().upper()
        if re.match(r"(?i)^status\s*:", line):
            result["status"] = line.split(":", 1)[1].strip()
    if result["symbol"] and result["status"]:
        return result
    return None


def is_new_signal(status_text: str) -> bool:
    return "new" in status_text.lower()


# ══════════════════════════════════════════════════════════════════════════════
# IBKR CLIENT
# ══════════════════════════════════════════════════════════════════════════════

ib = IB()

def ibkr_connect():
    util.startLoop()
    ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
    log.info(f"IBKR connected | accounts: {ib.wrapper.accounts}")


# ── Gate helpers ──────────────────────────────────────────────────────────────

def has_open_position(symbol: str) -> bool:
    """Gate 1 — existing IB position check."""
    result = any(p.contract.symbol == symbol and p.position != 0 for p in ib.positions())
    log.debug(f"Gate 1 | has_open_position({symbol}) => {result}")
    return result


def has_pending_order(symbol: str) -> bool:
    """Gate 2 — pending IB order check."""
    result = any(t.contract.symbol == symbol for t in ib.openTrades())
    log.debug(f"Gate 2 | has_pending_order({symbol}) => {result}")
    return result


def get_ask_price(contract) -> float | None:
    """Gate 4 — fetch live ask / last / close from IB market data."""
    ticker = ib.reqMktData(contract, "106", False, False)
    ib.sleep(3)
    ib.cancelMktData(contract)
    for label, val in [("ask", ticker.ask), ("last", ticker.last), ("close", ticker.close)]:
        if val and float(val) > 0:
            log.info(f"Gate 4 | {contract.symbol}: source='{label}'  ${float(val):.4f}")
            return float(val)
    log.warning(f"Gate 4 | {contract.symbol}: all price fields empty")
    return None


# ══════════════════════════════════════════════════════════════════════════════
# INDICATOR CALCULATIONS
# ══════════════════════════════════════════════════════════════════════════════

def calculate_ema(df: pd.DataFrame, period: int) -> pd.Series:
    return df["close"].ewm(span=period, adjust=False).mean()


def calculate_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    high_low   = df["high"] - df["low"]
    high_close = np.abs(df["high"] - df["close"].shift())
    low_close  = np.abs(df["low"]  - df["close"].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["ATR"]  = true_range.rolling(period).mean()
    return df


def calculate_ema_200(df: pd.DataFrame) -> pd.DataFrame:
    df["ema_200"] = calculate_ema(df, 200)
    return df


def calculate_session_vwap(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date_only"] = pd.to_datetime(df["date"]).dt.date
    today           = df["date_only"].max()
    mask            = df["date_only"] == today
    today_df        = df.loc[mask].copy()
    today_df["vwap"] = (
        (today_df["close"] * today_df["volume"]).cumsum() /
        today_df["volume"].cumsum()
    )
    df.loc[mask, "vwap"] = today_df["vwap"].values
    df["vwap"] = df["vwap"].ffill()
    return df


def calculate_ema_alignment_oscillator(df: pd.DataFrame,
                                       fast_period=8,
                                       mid_period=13,
                                       slow_period=14,
                                       trend_period=21,
                                       slope_threshold=1.0,
                                       spacing_min=0.2) -> pd.DataFrame:
    df["ema_fast"]  = calculate_ema(df, fast_period)
    df["ema_mid"]   = calculate_ema(df, mid_period)
    df["ema_slow"]  = calculate_ema(df, slow_period)
    df["ema_trend"] = calculate_ema(df, trend_period)
    df = calculate_atr(df, 14)

    def slope_deg(ema_s, atr_s):
        s = (ema_s - ema_s.shift(1)) / (atr_s + 0.0001) * 100
        return np.degrees(np.arctan(s))

    df["fast_angle"] = slope_deg(df["ema_fast"], df["ATR"])
    df["mid_angle"]  = slope_deg(df["ema_mid"],  df["ATR"])
    df["slow_angle"] = slope_deg(df["ema_slow"], df["ATR"])

    df["bullish_stack"] = (
        (df["ema_fast"]  > df["ema_mid"]) &
        (df["ema_mid"]   > df["ema_slow"]) &
        (df["ema_slow"]  > df["ema_trend"])
    )
    df["bearish_stack"] = (
        (df["ema_fast"]  < df["ema_mid"]) &
        (df["ema_mid"]   < df["ema_slow"]) &
        (df["ema_slow"]  < df["ema_trend"])
    )

    df["price_above_fast"] = df["close"] > df["ema_fast"]
    df["price_below_fast"] = df["close"] < df["ema_fast"]

    df["spacing_fast_mid"] = np.abs(df["ema_fast"] - df["ema_mid"]) / df["ema_mid"] * 100
    df["spacing_mid_slow"] = np.abs(df["ema_mid"]  - df["ema_slow"]) / df["ema_slow"] * 100
    df["avg_spacing"]      = (df["spacing_fast_mid"] + df["spacing_mid_slow"]) / 2

    df["volume_sma"]          = df["volume"].rolling(20).mean()
    df["volume_confirmation"] = df["volume"] > df["volume_sma"]

    df["trend_score"] = 0.0

    for i in range(1, len(df)):
        prev = df["trend_score"].iloc[i - 1]

        if df["bullish_stack"].iloc[i] and df["price_above_fast"].iloc[i]:
            avg_slope     = (df["fast_angle"].iloc[i] + df["mid_angle"].iloc[i] + df["slow_angle"].iloc[i]) / 3
            slope_bonus   = min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = 10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = 40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        elif df["bearish_stack"].iloc[i] and df["price_below_fast"].iloc[i]:
            avg_slope     = (abs(df["fast_angle"].iloc[i]) + abs(df["mid_angle"].iloc[i]) + abs(df["slow_angle"].iloc[i])) / 3
            slope_bonus   = -min(avg_slope * 3, 30)  if avg_slope > slope_threshold else 0
            spacing_bonus = -min(df["avg_spacing"].iloc[i] * 10, 20) if df["avg_spacing"].iloc[i] > spacing_min else 0
            vol_bonus     = -10 if df["volume_confirmation"].iloc[i] else 0
            new_score     = -40.0 + slope_bonus + spacing_bonus + vol_bonus
            df.loc[df.index[i], "trend_score"] = (new_score + prev) / 2

        else:
            df.loc[df.index[i], "trend_score"] = prev * 0.9

    df.loc[~df["volume_confirmation"], "trend_score"] *= 0.8
    df["trend_score"] = df["trend_score"].clip(-100, 100)

    # Threshold = 85
    df["strong_bull_signal"] = (
        (df["trend_score"] >= TREND_SCORE_ENTRY) &
        (df["trend_score"] > df["trend_score"].shift(1))
    )
    df["strong_bear_signal"] = (
        (df["trend_score"] <= -TREND_SCORE_ENTRY) &
        (df["trend_score"] < df["trend_score"].shift(1))
    )
    return df


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENT HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_document(symbol, entry_price, quantity, trend_score_at_entry):
    return {
        "instrument":        symbol,
        "direction":         "long",
        "entry_price":       entry_price,
        "highest_price":     entry_price,
        "quantity":          quantity,
        "entry_timestamp":   datetime.now(),
        "entry_trend_score": trend_score_at_entry,
        "order_id":          str(ObjectId()),
        "exit_price":        None,
        "exit_timestamp":    None,
        "reason":            None,
        "current_pnl":       0,
        "realized_pnl":      0,
        "status":            "open",
        "created_at":        datetime.now(),
        "updated_at":        datetime.now(),
    }


def update_trade_document(trade_id, updates: dict):
    updates["updated_at"] = datetime.now()
    trades_col.update_one({"_id": ObjectId(trade_id)}, {"$set": updates})


# ══════════════════════════════════════════════════════════════════════════════
# TRADING LOOP  (runs as asyncio task every 5 minutes)
# ══════════════════════════════════════════════════════════════════════════════

async def trading_loop():
    log.info("Trading loop started.")
    await asyncio.sleep(10)   # brief startup grace period

    while True:
        # ── EOD cleanup ───────────────────────────────────────────────────────
        watchlist_clear_eod()
        watchlist_print()

        # ── Load tickers dynamically from MongoDB ─────────────────────────────
        symbols = watchlist_get_all()

        if not symbols:
            log.info("Watchlist empty — nothing to evaluate. Waiting 5 min...")
            await asyncio.sleep(300)
            continue

        log.info(f"Evaluating {len(symbols)} ticker(s): {', '.join(symbols)}")

        # Qualify contracts fresh each cycle (watchlist may have changed)
        contracts = [Stock(s, "SMART", "USD") for s in symbols]
        try:
            ib.qualifyContracts(*contracts)
        except Exception as e:
            log.warning(f"qualifyContracts error: {e}")

        for contract in contracts:
            symbol = contract.symbol
            log.info(f"\n{'='*60}")
            log.info(f"Evaluating {symbol} at {datetime.now().strftime('%H:%M:%S')}")
            log.info(f"{'='*60}")

            # ── Fetch 5-min bars ──────────────────────────────────────────────
            try:
                bars = ib.reqHistoricalData(
                    contract,
                    endDateTime="",
                    durationStr="2 D",
                    barSizeSetting="5 mins",
                    whatToShow="TRADES",
                    useRTH=False,
                    formatDate=1,
                )
            except Exception as e:
                log.warning(f"{symbol}: reqHistoricalData error — {e}")
                continue

            if not bars:
                log.warning(f"{symbol}: no historical data returned")
                continue

            df = util.df(bars)
            df.columns = [c.lower() for c in df.columns]

            # ── Calculate indicators ──────────────────────────────────────────
            df = calculate_ema_alignment_oscillator(df)
            df = calculate_ema_200(df)
            df = calculate_session_vwap(df)

            row           = df.iloc[-1]
            current_price = row["close"]
            trend_score   = row["trend_score"]
            atr           = row.get("ATR", BASE_TRAILING_AMT)
            vwap          = row["vwap"]

            log.info(f"Price: ${current_price:.2f} | Score: {trend_score:.1f} | "
                     f"EMA200: ${row['ema_200']:.2f} | VWAP: ${vwap:.2f} | "
                     f"BullSig: {row['strong_bull_signal']}")

            # ── Check for open MongoDB trade ──────────────────────────────────
            open_trade = trades_col.find_one({"instrument": symbol, "status": "open"})

            # ═════════════════════════════════════════════════════════════════
            # SELL / POSITION MANAGEMENT
            # ═════════════════════════════════════════════════════════════════

            if open_trade:
                entry_price   = open_trade["entry_price"]
                highest_price = open_trade.get("highest_price", entry_price)
                quantity      = open_trade["quantity"]
                time_in_trade = (datetime.now() - open_trade["entry_timestamp"]).total_seconds() / 3600

                if current_price > highest_price:
                    highest_price = current_price
                    update_trade_document(open_trade["_id"], {"highest_price": highest_price})

                profit_pct  = (current_price - entry_price) / entry_price
                pnl_ps      = current_price - entry_price
                current_pnl = pnl_ps * quantity
                update_trade_document(open_trade["_id"], {"current_pnl": current_pnl})

                log.info(f"OPEN | Entry: ${entry_price:.2f} | P&L: {profit_pct:.2%} | Score: {trend_score:.1f}")

                # ── Stop loss ─────────────────────────────────────────────────
                should_exit = False
                exit_reason = None

                if profit_pct <= -STOP_LOSS_1:
                    if trend_score < TREND_SCORE_DROP_EXIT:
                        should_exit = True
                        exit_reason = "stop_loss_10pct_trend_weak"
                        log.info(f"EXIT: 10% loss + score {trend_score:.1f} < {TREND_SCORE_DROP_EXIT}")
                    elif profit_pct <= -STOP_LOSS_2:
                        should_exit = True
                        exit_reason = "stop_loss_15pct_hard"
                        log.info(f"EXIT: 15% hard stop reached")
                    else:
                        log.info(f"Holding at {profit_pct:.2%} loss — trend still strong ({trend_score:.1f})")

                if should_exit and quantity > 0:
                    ib.placeOrder(contract, MarketOrder("SELL", quantity))
                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           exit_reason,
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"STOP LOSS | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Take profit ───────────────────────────────────────────────
                if profit_pct >= TAKE_PROFIT_PCT:
                    ib.placeOrder(contract, MarketOrder("SELL", quantity))
                    realized = pnl_ps * quantity
                    update_trade_document(open_trade["_id"], {
                        "exit_price":       current_price,
                        "exit_timestamp":   datetime.now(),
                        "reason":           f"take_profit_{profit_pct:.2%}",
                        "realized_pnl":     realized,
                        "exit_trend_score": trend_score,
                        "status":           "closed",
                    })
                    log.info(f"TAKE PROFIT | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                    await asyncio.sleep(0.5)
                    continue

                # ── Trailing stop (only when profitable) ──────────────────────
                if profit_pct > 0:
                    atr_stop  = highest_price - (atr * ATR_MULTIPLIER)
                    pct_stop  = highest_price * (1 - MIN_TRAILING_PCT)
                    fix_stop  = highest_price - BASE_TRAILING_AMT
                    t_factor  = min(1.0, time_in_trade / 24)
                    trail_stop = max(atr_stop * (1 - 0.25 * t_factor), pct_stop, fix_stop)

                    if current_price <= trail_stop and quantity > 0:
                        ib.placeOrder(contract, MarketOrder("SELL", quantity))
                        realized = pnl_ps * quantity
                        update_trade_document(open_trade["_id"], {
                            "exit_price":       current_price,
                            "exit_timestamp":   datetime.now(),
                            "reason":           "trailing_stop_hit",
                            "realized_pnl":     realized,
                            "exit_trend_score": trend_score,
                            "status":           "closed",
                        })
                        log.info(f"TRAILING STOP | {symbol} @ ${current_price:.2f} | P&L: ${realized:.2f}")
                        await asyncio.sleep(0.5)
                        continue

            # ═════════════════════════════════════════════════════════════════
            # BUY / ENTRY LOGIC
            # Every iteration re-evaluates ALL tickers on the watchlist.
            # A ticker is only excluded AFTER a successful buy is placed —
            # not on the first failed evaluation (which was the original bug).
            # ═════════════════════════════════════════════════════════════════

            else:
                # ── Already bought this ticker today? Skip entirely. ───────────
                # excluded_col is written ONLY when a BUY order fires.
                # Tickers that fail entry conditions keep being re-checked
                # every 5-minute cycle until conditions are met or EOD clears
                # the watchlist.
                today_str     = datetime.now().date().isoformat()
                exclude_entry = excluded_col.find_one({"ticker": symbol, "date": today_str})
                if exclude_entry:
                    log.info(f"{symbol}: already bought today — skipping entry evaluation")
                    await asyncio.sleep(0.5)
                    continue

                # ── Entry condition checks ────────────────────────────────────
                above_ema200 = current_price > row["ema_200"]
                above_vwap   = current_price > vwap
                bull_signal  = bool(row["strong_bull_signal"])

                if not bull_signal:
                    log.info(f"{symbol}: no signal yet (score={trend_score:.1f}, need >={TREND_SCORE_ENTRY} & rising) — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue
                if not above_ema200:
                    log.info(f"{symbol}: below EMA200 (${current_price:.2f} < ${row['ema_200']:.2f}) — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue
                if not above_vwap:
                    log.info(f"{symbol}: below VWAP (${current_price:.2f} < ${vwap:.2f}) — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 1: no existing IB position ──────────────────────────
                if has_open_position(symbol):
                    log.info(f"{symbol}: Gate 1 FAILED — existing IB position — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 2: no pending IB order ───────────────────────────────
                if has_pending_order(symbol):
                    log.info(f"{symbol}: Gate 2 FAILED — pending IB order — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                # ── Gate 4: valid ask price ───────────────────────────────────
                ask_price = get_ask_price(contract)
                if not ask_price:
                    log.info(f"{symbol}: Gate 4 FAILED — no valid ask price — will re-check next cycle")
                    await asyncio.sleep(0.5)
                    continue

                # ── All gates passed → place order ────────────────────────────
                ib.placeOrder(contract, MarketOrder("BUY", ORDER_SIZE))

                trade_doc = create_trade_document(
                    symbol               = symbol,
                    entry_price          = ask_price,
                    quantity             = ORDER_SIZE,
                    trend_score_at_entry = trend_score,
                )
                trades_col.insert_one(trade_doc)

                # ── Mark as bought ONLY here, after a successful order ─────────
                # This is the fix: excluded_col.insert_one() was previously
                # called unconditionally before the gate checks, which caused
                # the ticker to be skipped on every subsequent cycle even when
                # no buy had actually occurred.
                excluded_col.insert_one({"ticker": symbol, "date": today_str})

                log.info(
                    f"BUY | {symbol} | ask=${ask_price:.2f} | qty={ORDER_SIZE} | "
                    f"score={trend_score:.1f} | EMA200 ✓ | VWAP ✓ | G1 ✓ | G2 ✓ | G4 ✓"
                )

            await asyncio.sleep(0.5)   # brief pause between tickers

        # ── End of scan cycle ─────────────────────────────────────────────────
        log.info(f"\n{'='*60}")
        log.info("Scan complete. Next scan in 5 minutes.")
        watchlist_print()
        log.info(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# TELEGRAM POLLER  (runs as asyncio task continuously)
# Adds new tickers to MongoDB watchlist_tickers collection.
# ══════════════════════════════════════════════════════════════════════════════

def load_session() -> str:
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, encoding="utf-8") as f:
            return f.read().strip()
    return ""


def save_session(s: str):
    with open(SESSION_FILE, "w", encoding="utf-8") as f:
        f.write(s)
    log.info(f"Telegram session saved → {SESSION_FILE}")


async def telegram_poller():
    log.info("Telegram poller started.")

    while True:
        tg = TelegramClient(StringSession(load_session()), API_ID, API_HASH)
        try:
            await tg.start(phone=PHONE)
            save_session(tg.session.save())

            # Join channel if needed
            m = re.search(r't\.me/(?:joinchat/|\+)([A-Za-z0-9_-]+)', INVITE_LINK)
            if not m:
                raise ValueError(f"Cannot parse invite hash: {INVITE_LINK}")
            try:
                await tg(ImportChatInviteRequest(m.group(1)))
                log.info("Joined Telegram channel")
            except UserAlreadyParticipantError:
                log.info("Already a member of the channel")

            channel      = await tg.get_entity(INVITE_LINK)
            last_seen_id = None
            log.info(f"Polling channel: {channel.title} (ID: {channel.id})")

            while True:
                messages = await tg.get_messages(channel, limit=1)
                if messages:
                    msg    = messages[0]
                    is_new = (last_seen_id is None or msg.id != last_seen_id)

                    if msg.text and is_new:
                        parsed = parse_structured_message(msg.text)
                        if parsed:
                            symbol = parsed["symbol"]
                            status = parsed["status"]
                        else:
                            symbol = extract_ticker(msg.text)
                            status = "NEW"

                        preview = msg.text[:120] + ("..." if len(msg.text) > 120 else "")
                        safe_print(f"\n[{msg.date}]")
                        safe_print(f"   Message : {preview}")
                        safe_print(f"   Ticker  : {symbol or '(not found)'}")
                        safe_print(f"   Status  : NEW")

                        if symbol and is_new_signal(status):
                            inserted = watchlist_add(symbol, source="telegram")
                            if inserted:
                                safe_print(f"   → Added {symbol} to watchlist_tickers in MongoDB")
                            else:
                                safe_print(f"   → {symbol} already in watchlist_tickers, skipped")
                        elif symbol:
                            log.info(f"{symbol}: status='{status}' — not NEW, skipping watchlist add")

                        last_seen_id = msg.id

                log.debug(f"Telegram: waiting {POLL_INTERVAL}s...")
                await asyncio.sleep(POLL_INTERVAL)

        except FloodWaitError as e:
            log.warning(f"Telegram rate-limited — waiting {e.seconds}s")
            await asyncio.sleep(e.seconds + 5)
        except KeyboardInterrupt:
            log.info("Telegram poller stopped by user.")
            break
        except Exception as e:
            log.error(f"Telegram error: {type(e).__name__}: {e} — reconnecting in 15s")
            await asyncio.sleep(15)
        finally:
            try:
                await tg.disconnect()
            except Exception:
                pass


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT — run both tasks concurrently on the same event loop
# ══════════════════════════════════════════════════════════════════════════════

async def main():
    log.info("=== Unified Trading Bot Starting ===")
    ibkr_connect()
    await asyncio.gather(
        telegram_poller(),   # continuously polls Telegram & updates watchlist_tickers
        trading_loop(),      # every 5 min reads watchlist_tickers & manages trades
    )


if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        log.info("Bot stopped by user.")
    finally:
        ib.disconnect()
        log.info("IBKR disconnected.")
